# E7 — Waterbirds Positive Real-Image Bridge
## Prospective eligibility gate + locked consistency sweep

This notebook is the final real-image bridge experiment for:

**When Consistency Misleads: Nuisance-Driven Partition Transitions in Unsupervised Clustering**

### Scientific role

E1–E3 already establish the mechanism in controlled population, sampled-augmentation, and learned-CNN settings.  
E4–E6 are boundary tests.  
**E7 asks whether the complete semantic → nuisance transition can occur in a natural-image representation.**

Waterbirds provides two separate variables:

- `y`: semantic bird class — landbird vs waterbird
- `place`: nuisance background — land background vs water background

The notebook is intentionally split into two stages.

### E7-A — eligibility only

Before any λ-sweep is examined, the frozen representation must show:

1. a stable semantic baseline;
2. a nuisance direction that is recoverable and competitive but weaker than semantics;
3. a **misaligned augmentation** that empirically disrupts semantic assignments more than nuisance assignments;
4. an **aligned control** with the opposite qualitative tendency.

If these prerequisites fail, the notebook records a **NO-GO** and does not run E7-B.

### E7-B — locked λ sweep

Only if E7-A passes do we evaluate the predeclared λ-grid.  
The desired mechanism is:

semantic baseline → oblique drift → nuisance takeover,

while semantic quality decreases and consistency increases.

### Important interpretation boundary

Semantic and nuisance labels are used to **define/audit the two-dimensional competition plane** and to evaluate the result. They are not supplied to the λ-dependent objective itself. This is a controlled mechanism experiment, not a claim that the full audit procedure is an unsupervised clustering algorithm.

In [1]:
# 1. Imports and frozen protocol

import os
import sys
import json
import math
import random
import hashlib
import zipfile
import platform
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFilter, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import transforms, models

import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, adjusted_rand_score

CFG = {
    "experiment_id": "E7_Waterbirds_real_image_bridge_v1",
    "seeds": [11, 22, 33, 44, 55],

    # Official Waterbirds metadata split:
    # 0=train, 1=val, 2=test.
    # We use train+val as DEVELOPMENT and keep test untouched.
    "development_splits": [0, 1],
    "test_split": 2,

    # Exact 2x2 balancing removes the original y-place correlation.
    "balance_by_y_place": True,

    # Backbone / preprocessing
    "backbone": "ImageNet_ResNet50",
    "image_size": 224,
    "batch_size": 64,
    "num_workers": 2,

    # Fixed λ-grid — locked before seeing E7 outcomes
    "lambda_grid": [0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8, 12, 16, 24],

    # Signed projective half-circle. +/-90° represent the nuisance line.
    "theta_grid_deg": list(np.arange(-90.0, 90.0 + 0.25, 0.25)),
    "collapse_angle_deg": 80.0,

    # Augmentation Monte Carlo
    "n_aug_views": 6,

    # E7-A frozen eligibility thresholds
    "semantic_acc_min": 0.75,
    "semantic_ari_min": 0.20,
    "nuisance_probe_min": 0.70,
    "nuisance_competitive_ratio_min": 0.45,
    "nuisance_competitive_ratio_max": 0.98,
    "baseline_abs_theta_max_deg": 20.0,
    "misalignment_margin": 0.015,
    "aligned_margin": 0.005,
    "eligibility_seed_fraction": 0.60,

    # Misaligned view: corrupt centered bird evidence while retaining much
    # of the peripheral background.
    "center_mask_frac_min": 0.34,
    "center_mask_frac_max": 0.54,
    "center_blur_radius": 4.0,

    # Aligned control: preserve central bird region and corrupt the border,
    # where much of the background evidence lies.
    "border_frac_min": 0.18,
    "border_frac_max": 0.30,
    "border_blur_radius": 6.0,

    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

ROOT = Path("/kaggle/working/E7_Waterbirds_real_image_bridge")
ROOT.mkdir(parents=True, exist_ok=True)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

with open(ROOT / "config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("Device:", CFG["device"])
print("Output directory:", ROOT)

Device: cuda
Output directory: /kaggle/working/E7_Waterbirds_real_image_bridge


## 2. Auto-detect Waterbirds under `/kaggle/input`

The official metadata contains at least:

- `img_filename`
- `y`
- `place`
- `split`

The notebook searches all attached Kaggle inputs and validates the metadata/image paths instead of assuming a hard-coded dataset directory.

In [2]:
REQUIRED_META = {"img_filename", "y", "place", "split"}

def locate_waterbirds():
    candidates = []

    for root, dirs, files in os.walk("/kaggle/input"):
        if "metadata.csv" in files:
            meta_path = Path(root) / "metadata.csv"
            try:
                df = pd.read_csv(meta_path)
            except Exception:
                continue

            if REQUIRED_META.issubset(df.columns):
                candidates.append((meta_path, df))

    if not candidates:
        raise FileNotFoundError(
            "No Waterbirds metadata.csv with columns "
            "img_filename, y, place, split was found under /kaggle/input.\n"
            "Attach the Waterbirds Kaggle dataset first, then rerun."
        )

    # Prefer a candidate whose first few referenced images can be resolved.
    for meta_path, df in candidates:
        meta_dir = meta_path.parent

        possible_roots = [
            meta_dir,
            meta_dir.parent,
            meta_dir / "waterbird_complete95_forest2water2",
        ]

        for data_root in possible_roots:
            ok = 0
            for rel in df["img_filename"].astype(str).head(20):
                if (data_root / rel).exists():
                    ok += 1
            if ok >= min(5, len(df.head(20))):
                return meta_path, data_root, df

    raise RuntimeError(
        "A Waterbirds-like metadata.csv was found, but image paths could not "
        "be resolved. Print the Kaggle input directory tree and inspect the layout."
    )

META_PATH, DATA_ROOT, metadata = locate_waterbirds()

print("metadata.csv:", META_PATH)
print("image root:", DATA_ROOT)
print("rows:", len(metadata))
print("columns:", list(metadata.columns))

print("\nSplit counts:")
display(metadata["split"].value_counts().sort_index().rename("count").to_frame())

print("\ny x place group counts:")
display(pd.crosstab(metadata["y"], metadata["place"]))

# Validate labels.
assert set(metadata["y"].unique()).issubset({0, 1})
assert set(metadata["place"].unique()).issubset({0, 1})
assert set(metadata["split"].unique()).issubset({0, 1, 2})

# Validate every referenced path.
missing = []
for rel in metadata["img_filename"].astype(str):
    if not (DATA_ROOT / rel).exists():
        missing.append(rel)
        if len(missing) >= 10:
            break

if missing:
    raise RuntimeError(f"Some Waterbirds images are missing. Examples: {missing}")

print("\nWaterbirds metadata/image-path validation passed.")

metadata.csv: /kaggle/input/datasets/satwiksps/waterbirds-spurious-correlation-benchmark/waterbird_complete95_forest2water2/metadata.csv
image root: /kaggle/input/datasets/satwiksps/waterbirds-spurious-correlation-benchmark/waterbird_complete95_forest2water2
rows: 11788
columns: ['img_id', 'img_filename', 'y', 'split', 'place', 'place_filename']

Split counts:


,count
split,
0,4795
1,1199
2,5794



y x place group counts:


place,0,1
y,,
0,6220,2905
1,831,1832



Waterbirds metadata/image-path validation passed.


## 3. Build balanced development and held-out test audit sets

The original Waterbirds distribution deliberately correlates bird type and background.  
That correlation would confound our two-factor mechanism test.

For each seed, we therefore create an **exactly balanced 2×2 audit sample** across:

- landbird + land background
- landbird + water background
- waterbird + land background
- waterbird + water background

Development uses official train+validation only.  
Official test remains untouched until final evaluation.

In [3]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def balanced_group_indices(df, allowed_splits, seed):
    rng = np.random.default_rng(seed)
    sub = df[df["split"].isin(allowed_splits)].copy()

    groups = {}
    counts = {}

    for y in [0, 1]:
        for place in [0, 1]:
            idx = sub[(sub["y"] == y) & (sub["place"] == place)].index.to_numpy()
            groups[(y, place)] = idx
            counts[(y, place)] = len(idx)

    n = min(counts.values())
    if n < 25:
        raise RuntimeError(
            f"Smallest y×place group has only {n} examples. "
            "This Waterbirds copy/split is too small for the frozen E7 protocol."
        )

    chosen = []
    for key in [(0,0), (0,1), (1,0), (1,1)]:
        idx = groups[key].copy()
        rng.shuffle(idx)
        chosen.extend(idx[:n].tolist())

    chosen = np.asarray(chosen, dtype=np.int64)
    rng.shuffle(chosen)

    return chosen, counts, n

for seed in CFG["seeds"]:
    dev_idx, dev_counts, dev_n = balanced_group_indices(
        metadata, CFG["development_splits"], seed
    )
    test_idx, test_counts, test_n = balanced_group_indices(
        metadata, [CFG["test_split"]], seed + 10000
    )

    print(
        f"seed={seed}: dev={len(dev_idx)} ({dev_n}/group), "
        f"test={len(test_idx)} ({test_n}/group)"
    )

seed=11: dev=756 (189/group), test=2568 (642/group)
seed=22: dev=756 (189/group), test=2568 (642/group)
seed=33: dev=756 (189/group), test=2568 (642/group)
seed=44: dev=756 (189/group), test=2568 (642/group)
seed=55: dev=756 (189/group), test=2568 (642/group)


## 4. Waterbirds dataset wrapper and ImageNet preprocessing

In [4]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

base_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(CFG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class WaterbirdsRows(Dataset):
    def __init__(self, metadata_df, indices, data_root, transform=None):
        self.df = metadata_df.loc[indices].reset_index(drop=False)
        self.data_root = Path(data_root)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        path = self.data_root / str(row["img_filename"])
        img = Image.open(path).convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return (
            img,
            int(row["y"]),       # semantic: landbird/waterbird
            int(row["place"]),   # nuisance: land/water background
            int(row["index"]),
        )

print("Dataset wrapper ready.")

Dataset wrapper ready.


## 5. Frozen ImageNet ResNet-50 representation

No fine-tuning is performed.  
The final classifier is removed and the 2048-D penultimate representation is used.

In [5]:
try:
    weights = models.ResNet50_Weights.DEFAULT
    backbone = models.resnet50(weights=weights)
except Exception as e:
    raise RuntimeError(
        "Could not load pretrained ImageNet ResNet-50 weights. "
        "Do not replace with random weights because that changes the frozen protocol."
    ) from e

backbone.fc = nn.Identity()
backbone = backbone.to(CFG["device"]).eval()

for p in backbone.parameters():
    p.requires_grad_(False)

print("Backbone ready: frozen ImageNet ResNet-50.")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 245MB/s]


Backbone ready: frozen ImageNet ResNet-50.


## 6. Feature/audit utilities

The semantic axis is the development-set difference between waterbird and landbird means.

The nuisance axis is the water-background vs land-background mean difference **orthogonalized against the semantic axis**.

This ensures the audit plane separates semantic and nuisance competition rather than merely reflecting their correlation.

In [6]:
@torch.no_grad()
def encode(dataset):
    loader = DataLoader(
        dataset,
        batch_size=CFG["batch_size"],
        shuffle=False,
        num_workers=CFG["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )

    feats, ys, ps, ids = [], [], [], []

    for x, y, p, idx in loader:
        x = x.to(CFG["device"], non_blocking=True)
        h = backbone(x).detach().cpu().numpy()

        feats.append(h)
        ys.append(np.asarray(y))
        ps.append(np.asarray(p))
        ids.append(np.asarray(idx))

    return (
        np.concatenate(feats),
        np.concatenate(ys).astype(np.int64),
        np.concatenate(ps).astype(np.int64),
        np.concatenate(ids).astype(np.int64),
    )

def standardize_fit_apply(X):
    mean = X.mean(axis=0, keepdims=True)
    std = X.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    return (X - mean) / std, mean, std

def standardize_apply(X, mean, std):
    return (X - mean) / std

def unit(v):
    n = np.linalg.norm(v)
    if n < 1e-12:
        raise RuntimeError("Degenerate axis encountered.")
    return v / n

def make_plane(F, y, place):
    raw_sem = F[y == 1].mean(axis=0) - F[y == 0].mean(axis=0)
    raw_nui = F[place == 1].mean(axis=0) - F[place == 0].mean(axis=0)

    ws = unit(raw_sem)
    raw_nui_u = unit(raw_nui)

    raw_cosine = float(np.dot(ws, raw_nui_u))

    nuisance_resid = raw_nui - np.dot(raw_nui, ws) * ws
    residual_fraction = float(
        np.linalg.norm(nuisance_resid) /
        (np.linalg.norm(raw_nui) + 1e-12)
    )
    wn = unit(nuisance_resid)

    return ws, wn, raw_cosine, residual_fraction

def fit_threshold(q):
    return float(np.median(q))

def orientation_free_acc(pred, target):
    a = accuracy_score(target, pred)
    return max(a, 1.0 - a)

def metrics_from_projection(q, y, place, threshold):
    centered = q - threshold
    pred = (centered >= 0).astype(np.int64)

    # Empirical clusterability surrogate used consistently with the bridge protocol.
    M = float(
        (np.mean(np.abs(centered)) ** 2) /
        (np.mean(centered ** 2) + 1e-12)
    )

    return {
        "M": M,
        "semantic_acc": float(orientation_free_acc(pred, y)),
        "nuisance_acc": float(orientation_free_acc(pred, place)),
        "semantic_ari": float(adjusted_rand_score(y, pred)),
        "nuisance_ari": float(adjusted_rand_score(place, pred)),
    }

def linear_probe_train_internal(F, target, seed):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(target))
    rng.shuffle(idx)

    # Fixed 75/25 internal split, stratified by target using explicit per-class split.
    tr_idx, va_idx = [], []
    for cls in [0, 1]:
        c = idx[target[idx] == cls]
        cut = int(round(0.75 * len(c)))
        tr_idx.extend(c[:cut].tolist())
        va_idx.extend(c[cut:].tolist())

    clf = LogisticRegression(
        max_iter=4000,
        class_weight="balanced",
        random_state=seed,
    )
    clf.fit(F[tr_idx], target[tr_idx])
    return float(clf.score(F[va_idx], target[va_idx]))

def coords(F, ws, wn):
    return F @ ws, F @ wn

def projection_from_coords(s, n, theta_deg):
    t = np.deg2rad(theta_deg)
    return np.cos(t) * s + np.sin(t) * n

print("Audit utilities ready.")

Audit utilities ready.


## 7. Frozen image interventions

### Misaligned augmentation — central semantic corruption

Waterbirds images are cropped/centered on the bird.  
The misaligned transform therefore perturbs the central region strongly while preserving most peripheral background evidence.

### Aligned control — peripheral background corruption

The control preserves the central region while blurring/neutralizing a random border band.

The notebook does **not assume** these transforms have the intended effect.  
E7-A directly measures assignment disagreement along the semantic and nuisance endpoints.

In [7]:
def resize_center_crop(img):
    return transforms.functional.center_crop(
        transforms.functional.resize(img, [256, 256]),
        [CFG["image_size"], CFG["image_size"]]
    )

def tensor_normalize(img):
    return transforms.functional.normalize(
        transforms.functional.to_tensor(img),
        IMAGENET_MEAN,
        IMAGENET_STD,
    )

class MisalignedCentralCorruption:
    def __call__(self, img):
        img = resize_center_crop(img)
        w, h = img.size

        frac = random.uniform(
            CFG["center_mask_frac_min"],
            CFG["center_mask_frac_max"]
        )

        mw = int(round(w * frac))
        mh = int(round(h * frac))

        jitter_x = int(round(w * 0.05))
        jitter_y = int(round(h * 0.05))

        cx = w // 2 + random.randint(-jitter_x, jitter_x)
        cy = h // 2 + random.randint(-jitter_y, jitter_y)

        x0 = max(0, cx - mw // 2)
        y0 = max(0, cy - mh // 2)
        x1 = min(w, x0 + mw)
        y1 = min(h, y0 + mh)

        # Strong blur rather than an artificial solid rectangle.
        blurred = img.filter(ImageFilter.GaussianBlur(CFG["center_blur_radius"]))
        out = img.copy()
        out.paste(blurred.crop((x0, y0, x1, y1)), (x0, y0))

        # Mild geometry jitter, applied globally.
        if random.random() < 0.5:
            out = transforms.functional.hflip(out)

        return tensor_normalize(out)

class AlignedPeripheralCorruption:
    def __call__(self, img):
        img = resize_center_crop(img)
        w, h = img.size

        frac = random.uniform(
            CFG["border_frac_min"],
            CFG["border_frac_max"]
        )
        b = int(round(min(w, h) * frac))

        blurred = img.filter(ImageFilter.GaussianBlur(CFG["border_blur_radius"]))
        out = blurred.copy()

        # Restore the central bird region from the clean image.
        out.paste(
            img.crop((b, b, w - b, h - b)),
            (b, b)
        )

        if random.random() < 0.5:
            out = transforms.functional.hflip(out)

        return tensor_normalize(out)

MISALIGNED_AUG = MisalignedCentralCorruption()
ALIGNED_AUG = AlignedPeripheralCorruption()

class WaterbirdsAugRows(Dataset):
    def __init__(self, metadata_df, indices, data_root, aug, view_seed):
        self.df = metadata_df.loc[indices].reset_index(drop=False)
        self.data_root = Path(data_root)
        self.aug = aug
        self.view_seed = int(view_seed)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.data_root / str(row["img_filename"])).convert("RGB")

        local_seed = self.view_seed * 1000003 + int(i)
        random.seed(local_seed)
        torch.manual_seed(local_seed)

        x = self.aug(img)

        return (
            x,
            int(row["y"]),
            int(row["place"]),
            int(row["index"]),
        )

print("Augmentations ready.")

Augmentations ready.


## 8. E7-A — eligibility gate

For each seed:

1. exact 2×2 group balance;
2. frozen ResNet-50 features;
3. training-only standardization;
4. semantic and orthogonal nuisance axes;
5. λ=0 optimum searched over the complete signed semantic–nuisance plane;
6. nuisance recoverability probe;
7. endpoint disagreement under misaligned and aligned transforms.

### Frozen eligibility conditions

A seed passes only if:

- semantic endpoint clusterability exceeds nuisance endpoint clusterability;
- semantic endpoint accuracy ≥ 0.75;
- semantic endpoint ARI ≥ 0.20;
- nuisance linear-probe accuracy ≥ 0.70;
- `M_nui / M_sem ∈ [0.45, 0.98]`;
- λ=0 selected `|θ| ≤ 20°`;
- misaligned: `R_sem - R_nui ≥ 0.015`;
- aligned: `R_nui - R_sem ≥ 0.005`.

The entire E7 experiment is eligible only if at least 60% of the five seeds pass.

These conditions are evaluated before the λ-path is inspected.

In [8]:
def endpoint_disagreement(base_coords, aug_coords_list):
    s, n = base_coords
    ts = fit_threshold(s)
    tn = fit_threshold(n)

    base_s_pred = (s - ts >= 0).astype(np.int8)
    base_n_pred = (n - tn >= 0).astype(np.int8)

    sem_vals = []
    nui_vals = []

    for aug_s, aug_n in aug_coords_list:
        sem_vals.append(
            np.mean(base_s_pred != (aug_s - ts >= 0).astype(np.int8))
        )
        nui_vals.append(
            np.mean(base_n_pred != (aug_n - tn >= 0).astype(np.int8))
        )

    return float(np.mean(sem_vals)), float(np.mean(nui_vals))

def baseline_theta_search(base_coords):
    s, n = base_coords
    rows = []

    for theta in CFG["theta_grid_deg"]:
        q = projection_from_coords(s, n, theta)
        th = fit_threshold(q)
        centered = q - th

        M = float(
            (np.mean(np.abs(centered)) ** 2) /
            (np.mean(centered ** 2) + 1e-12)
        )

        D = 1.0 - M
        rows.append((theta, abs(theta), M, D, th))

    df = pd.DataFrame(
        rows,
        columns=["theta_deg", "abs_theta_deg", "M", "D", "threshold"]
    )

    best = df.sort_values(
        ["D", "abs_theta_deg", "theta_deg"],
        ascending=[True, True, True]
    ).iloc[0]

    return df, best

def prepare_seed(seed):
    dev_idx, dev_counts, dev_n = balanced_group_indices(
        metadata, CFG["development_splits"], seed
    )
    test_idx, test_counts, test_n = balanced_group_indices(
        metadata, [CFG["test_split"]], seed + 10000
    )

    dev_ds = WaterbirdsRows(metadata, dev_idx, DATA_ROOT, base_transform)
    test_ds = WaterbirdsRows(metadata, test_idx, DATA_ROOT, base_transform)

    dev_raw, ydev, pdev, dev_ids = encode(dev_ds)
    test_raw, ytest, ptest, test_ids = encode(test_ds)

    devF, meanF, stdF = standardize_fit_apply(dev_raw)
    testF = standardize_apply(test_raw, meanF, stdF)

    ws, wn, raw_cos, residual_frac = make_plane(devF, ydev, pdev)

    dev_coords = coords(devF, ws, wn)
    test_coords = coords(testF, ws, wn)

    return {
        "seed": seed,
        "dev_idx": dev_idx,
        "test_idx": test_idx,
        "dev_n_per_group": dev_n,
        "test_n_per_group": test_n,
        "devF": devF,
        "testF": testF,
        "ydev": ydev,
        "pdev": pdev,
        "ytest": ytest,
        "ptest": ptest,
        "meanF": meanF,
        "stdF": stdF,
        "ws": ws,
        "wn": wn,
        "dev_coords": dev_coords,
        "test_coords": test_coords,
        "raw_axis_cosine": raw_cos,
        "nuisance_residual_fraction": residual_frac,
    }

def augmented_coords_for_seed(base, aug_name):
    aug = MISALIGNED_AUG if aug_name == "misaligned" else ALIGNED_AUG
    out = []

    for v in range(CFG["n_aug_views"]):
        aug_ds = WaterbirdsAugRows(
            metadata,
            base["dev_idx"],
            DATA_ROOT,
            aug,
            view_seed=base["seed"] * 100 + v + 1,
        )
        aug_raw, _, _, _ = encode(aug_ds)
        augF = standardize_apply(
            aug_raw,
            base["meanF"],
            base["stdF"],
        )
        out.append(coords(augF, base["ws"], base["wn"]))

    return out

eligibility_rows = []
BASE_CACHE = {}
AUG_CACHE = {}

for seed in CFG["seeds"]:
    print(f"\nE7-A seed={seed}")

    base = prepare_seed(seed)
    BASE_CACHE[seed] = base

    s, n = base["dev_coords"]

    sem_th = fit_threshold(s)
    nui_th = fit_threshold(n)

    sem_m = metrics_from_projection(
        s, base["ydev"], base["pdev"], sem_th
    )
    nui_m = metrics_from_projection(
        n, base["ydev"], base["pdev"], nui_th
    )

    nuisance_probe = linear_probe_train_internal(
        base["devF"], base["pdev"], seed
    )

    ratio = nui_m["M"] / max(sem_m["M"], 1e-12)

    _, baseline_best = baseline_theta_search(base["dev_coords"])
    baseline_theta = float(baseline_best["theta_deg"])

    mis_coords = augmented_coords_for_seed(base, "misaligned")
    ali_coords = augmented_coords_for_seed(base, "aligned")

    AUG_CACHE[(seed, "misaligned")] = mis_coords
    AUG_CACHE[(seed, "aligned")] = ali_coords

    mis_R_sem, mis_R_nui = endpoint_disagreement(
        base["dev_coords"], mis_coords
    )
    ali_R_sem, ali_R_nui = endpoint_disagreement(
        base["dev_coords"], ali_coords
    )

    checks = {
        "semantic_stronger": sem_m["M"] > nui_m["M"],
        "semantic_acc_ok": sem_m["semantic_acc"] >= CFG["semantic_acc_min"],
        "semantic_ari_ok": sem_m["semantic_ari"] >= CFG["semantic_ari_min"],
        "nuisance_probe_ok": nuisance_probe >= CFG["nuisance_probe_min"],
        "competitive_ratio_ok": (
            CFG["nuisance_competitive_ratio_min"]
            <= ratio
            <= CFG["nuisance_competitive_ratio_max"]
        ),
        "baseline_theta_ok": (
            abs(baseline_theta)
            <= CFG["baseline_abs_theta_max_deg"]
        ),
        "misalignment_ok": (
            mis_R_sem - mis_R_nui
            >= CFG["misalignment_margin"]
        ),
        "aligned_control_ok": (
            ali_R_nui - ali_R_sem
            >= CFG["aligned_margin"]
        ),
    }

    seed_eligible = all(checks.values())

    eligibility_rows.append({
        "seed": seed,
        "dev_n_per_group": base["dev_n_per_group"],
        "test_n_per_group": base["test_n_per_group"],
        "M_sem": sem_m["M"],
        "M_nui": nui_m["M"],
        "M_nui_over_M_sem": ratio,
        "semantic_endpoint_acc": sem_m["semantic_acc"],
        "semantic_endpoint_ari": sem_m["semantic_ari"],
        "nuisance_endpoint_acc": nui_m["nuisance_acc"],
        "nuisance_endpoint_ari": nui_m["nuisance_ari"],
        "nuisance_probe_internal_acc": nuisance_probe,
        "baseline_theta_deg": baseline_theta,
        "baseline_abs_theta_deg": abs(baseline_theta),
        "mis_R_sem": mis_R_sem,
        "mis_R_nui": mis_R_nui,
        "mis_R_sem_minus_R_nui": mis_R_sem - mis_R_nui,
        "aligned_R_sem": ali_R_sem,
        "aligned_R_nui": ali_R_nui,
        "aligned_R_nui_minus_R_sem": ali_R_nui - ali_R_sem,
        "raw_axis_cosine": base["raw_axis_cosine"],
        "nuisance_residual_fraction": base["nuisance_residual_fraction"],
        **checks,
        "seed_eligible": seed_eligible,
    })

eligibility = pd.DataFrame(eligibility_rows)
eligibility.to_csv(ROOT / "E7A_eligibility_by_seed.csv", index=False)

display(eligibility)

pass_fraction = float(eligibility["seed_eligible"].mean())
E7_ELIGIBLE = pass_fraction >= CFG["eligibility_seed_fraction"]

eligibility_summary = {
    "eligible": bool(E7_ELIGIBLE),
    "seed_pass_fraction": pass_fraction,
    "required_fraction": CFG["eligibility_seed_fraction"],
    "n_pass": int(eligibility["seed_eligible"].sum()),
    "n_total": len(eligibility),
}

with open(ROOT / "E7A_eligibility_summary.json", "w") as f:
    json.dump(eligibility_summary, f, indent=2)

print("\nE7-A ELIGIBILITY SUMMARY")
print(json.dumps(eligibility_summary, indent=2))

if not E7_ELIGIBLE:
    print(
        "\nE7-B WILL NOT RUN. "
        "The frozen real-image prerequisites were not met. "
        "Do not loosen thresholds after inspecting this result."
    )
else:
    print("\nE7-A PASSED. The locked λ-sweep is now authorized.")


E7-A seed=11

E7-A seed=22

E7-A seed=33

E7-A seed=44

E7-A seed=55


,seed,dev_n_per_group,test_n_per_group,M_sem,M_nui,M_nui_over_M_sem,semantic_endpoint_acc,semantic_endpoint_ari,nuisance_endpoint_acc,nuisance_endpoint_ari,...,nuisance_residual_fraction,semantic_stronger,semantic_acc_ok,semantic_ari_ok,nuisance_probe_ok,competitive_ratio_ok,baseline_theta_ok,misalignment_ok,aligned_control_ok,seed_eligible
0,11,189,642,0.773766,0.630435,0.814762,0.891534,0.612684,0.917989,0.698461,...,0.986606,True,True,True,True,True,True,False,False,False
1,22,189,642,0.772482,0.672474,0.870537,0.894180,0.621009,0.936508,0.761841,...,0.982150,True,True,True,True,True,True,True,False,False
2,33,189,642,0.769404,0.646886,0.840762,0.896825,0.629391,0.931217,0.743452,...,0.987390,True,True,True,True,True,True,False,False,False
3,44,189,642,0.777173,0.677205,0.871370,0.902116,0.646322,0.944444,0.789845,...,0.987424,True,True,True,True,True,True,True,False,False
4,55,189,642,0.772744,0.665646,0.861405,0.894180,0.621009,0.931217,0.743452,...,0.988355,True,True,True,True,True,True,True,False,False



E7-A ELIGIBILITY SUMMARY
{
  "eligible": false,
  "seed_pass_fraction": 0.0,
  "required_fraction": 0.6,
  "n_pass": 0,
  "n_total": 5
}

E7-B WILL NOT RUN. The frozen real-image prerequisites were not met. Do not loosen thresholds after inspecting this result.


## 9. E7-B — locked consistency sweep

This block executes **only if E7-A passes**.

For each θ:

\[
L_\lambda(\theta)=D(\theta)+\lambda R(\theta)
\]

where:

- `D(θ) = 1 - M(θ)` is the empirical clusterability distortion;
- `R(θ)` is augmentation disagreement on the development set;
- the projection threshold is always fit from the unaugmented development projection;
- the held-out official test set is used only after θ is selected.

The λ=0 solution is independent of augmentation by construction.

In [9]:
def build_theta_curve(base_coords, aug_coords_list):
    s, n = base_coords
    rows = []

    for theta in CFG["theta_grid_deg"]:
        q = projection_from_coords(s, n, theta)
        th = fit_threshold(q)
        centered = q - th

        M = float(
            (np.mean(np.abs(centered)) ** 2) /
            (np.mean(centered ** 2) + 1e-12)
        )
        D = 1.0 - M

        base_pred = (centered >= 0).astype(np.int8)
        disagree = []

        for aug_s, aug_n in aug_coords_list:
            q_aug = projection_from_coords(aug_s, aug_n, theta)
            aug_pred = (q_aug - th >= 0).astype(np.int8)
            disagree.append(np.mean(base_pred != aug_pred))

        R = float(np.mean(disagree))

        rows.append({
            "theta_deg": float(theta),
            "abs_theta_deg": float(abs(theta)),
            "threshold": th,
            "M": M,
            "D": D,
            "R": R,
        })

    return pd.DataFrame(rows)

def choose_theta(curve, lam):
    x = curve.copy()
    x["L"] = x["D"] + float(lam) * x["R"]

    # If numerically tied, prefer the smaller absolute rotation.
    return x.sort_values(
        ["L", "abs_theta_deg", "theta_deg"],
        ascending=[True, True, True]
    ).iloc[0]

e7_rows = []
curve_rows = []

if E7_ELIGIBLE:
    for seed in CFG["seeds"]:
        base = BASE_CACHE[seed]

        for aug_name in ["misaligned", "aligned"]:
            print(f"E7-B | seed={seed} | {aug_name}")

            curve = build_theta_curve(
                base["dev_coords"],
                AUG_CACHE[(seed, aug_name)],
            )

            curve["seed"] = seed
            curve["augmentation"] = aug_name
            curve_rows.append(curve)

            for lam in CFG["lambda_grid"]:
                best = choose_theta(curve, lam)

                theta = float(best["theta_deg"])
                th = float(best["threshold"])

                dev_q = projection_from_coords(
                    base["dev_coords"][0],
                    base["dev_coords"][1],
                    theta,
                )
                test_q = projection_from_coords(
                    base["test_coords"][0],
                    base["test_coords"][1],
                    theta,
                )

                dev_metrics = metrics_from_projection(
                    dev_q,
                    base["ydev"],
                    base["pdev"],
                    th,
                )
                test_metrics = metrics_from_projection(
                    test_q,
                    base["ytest"],
                    base["ptest"],
                    th,
                )

                e7_rows.append({
                    "seed": seed,
                    "augmentation": aug_name,
                    "lambda": lam,
                    "theta_deg": theta,
                    "abs_theta_deg": abs(theta),
                    "collapse_80deg": float(
                        abs(theta) >= CFG["collapse_angle_deg"]
                    ),
                    "objective_L": float(best["L"]),
                    "D": float(best["D"]),
                    "R": float(best["R"]),
                    "agreement": float(1.0 - best["R"]),
                    "dev_semantic_acc": dev_metrics["semantic_acc"],
                    "dev_nuisance_acc": dev_metrics["nuisance_acc"],
                    "dev_semantic_ari": dev_metrics["semantic_ari"],
                    "dev_nuisance_ari": dev_metrics["nuisance_ari"],
                    "test_semantic_acc": test_metrics["semantic_acc"],
                    "test_nuisance_acc": test_metrics["nuisance_acc"],
                    "test_semantic_ari": test_metrics["semantic_ari"],
                    "test_nuisance_ari": test_metrics["nuisance_ari"],
                })

    e7_results = pd.DataFrame(e7_rows)
    e7_curves = pd.concat(curve_rows, ignore_index=True)

    e7_results.to_csv(ROOT / "E7B_seed_level_results.csv", index=False)
    e7_curves.to_csv(ROOT / "E7B_theta_curves.csv", index=False)

    print("\nE7-B completed.")
    display(e7_results.head())
else:
    print("E7-B skipped because E7-A did not pass.")

E7-B skipped because E7-A did not pass.


## 10. Integrity checks, summary, and final E7 verdict

A positive E7 result requires, under **misaligned augmentation**:

- semantic baseline at λ=0;
- selected direction moves toward nuisance;
- semantic test quality decreases;
- nuisance test quality increases;
- nuisance-collapse probability reaches at least 0.5;
- augmentation agreement increases from λ=0 to the strongest λ.

The aligned control must remain below 0.5 nuisance-collapse probability across the tested λ-grid.

This is intentionally stricter than simply observing one noisy endpoint.

In [10]:
if E7_ELIGIBLE:
    # λ=0 must be identical for aligned/misaligned.
    zero = e7_results[e7_results["lambda"] == 0]

    for seed, g in zero.groupby("seed"):
        if len(g) != 2:
            raise RuntimeError(f"Missing λ=0 control row for seed={seed}")

        t = g["theta_deg"].to_numpy()
        if not np.allclose(t[0], t[1], atol=1e-12):
            raise RuntimeError(
                f"Integrity error: λ=0 differs by augmentation for seed={seed}: {t}"
            )

    summary = (
        e7_results
        .groupby(["augmentation", "lambda"], as_index=False)
        .agg(
            median_theta_deg=("theta_deg", "median"),
            median_abs_theta_deg=("abs_theta_deg", "median"),
            mean_abs_theta_deg=("abs_theta_deg", "mean"),
            p_nuisance_collapse=("collapse_80deg", "mean"),
            mean_agreement=("agreement", "mean"),
            mean_test_semantic_acc=("test_semantic_acc", "mean"),
            std_test_semantic_acc=("test_semantic_acc", "std"),
            mean_test_nuisance_acc=("test_nuisance_acc", "mean"),
            std_test_nuisance_acc=("test_nuisance_acc", "std"),
            mean_test_semantic_ari=("test_semantic_ari", "mean"),
            mean_test_nuisance_ari=("test_nuisance_ari", "mean"),
            n_seeds=("seed", "nunique"),
        )
    )

    summary.to_csv(ROOT / "E7B_summary.csv", index=False)
    display(summary)

    mis = summary[
        summary["augmentation"] == "misaligned"
    ].sort_values("lambda")
    ali = summary[
        summary["augmentation"] == "aligned"
    ].sort_values("lambda")

    mis0 = mis.iloc[0]
    mismax = mis.iloc[-1]

    checks = {
        "E7A_eligible": True,
        "baseline_semantic": bool(
            mis0["median_abs_theta_deg"]
            <= CFG["baseline_abs_theta_max_deg"]
            and mis0["mean_test_semantic_acc"]
            >= CFG["semantic_acc_min"]
        ),
        "angle_moves_toward_nuisance": bool(
            mismax["median_abs_theta_deg"]
            > mis0["median_abs_theta_deg"]
        ),
        "semantic_accuracy_decreases": bool(
            mismax["mean_test_semantic_acc"]
            < mis0["mean_test_semantic_acc"]
        ),
        "nuisance_accuracy_increases": bool(
            mismax["mean_test_nuisance_acc"]
            > mis0["mean_test_nuisance_acc"]
        ),
        "misaligned_nuisance_takeover": bool(
            mis["p_nuisance_collapse"].max() >= 0.5
        ),
        "agreement_increases": bool(
            mismax["mean_agreement"] > mis0["mean_agreement"]
        ),
        "aligned_control_safe_all_lambda": bool(
            (ali["p_nuisance_collapse"] < 0.5).all()
        ),
    }

    E7_GO = bool(all(checks.values()))

    transition_rows = mis[
        mis["p_nuisance_collapse"] >= 0.5
    ]
    first_transition_lambda = (
        None
        if len(transition_rows) == 0
        else float(transition_rows.iloc[0]["lambda"])
    )

    verdict = {
        "experiment": CFG["experiment_id"],
        "go": E7_GO,
        "first_lambda_pcollapse_ge_0_5": first_transition_lambda,
        "checks": checks,
        "interpretation": (
            "Positive real-image bridge: full mechanism supported."
            if E7_GO
            else
            "E7-A passed, but the locked λ-sweep did not show the complete takeover mechanism."
        ),
    }

else:
    summary = None
    E7_GO = False

    verdict = {
        "experiment": CFG["experiment_id"],
        "go": False,
        "first_lambda_pcollapse_ge_0_5": None,
        "checks": {
            "E7A_eligible": False,
        },
        "interpretation": (
            "Waterbirds did not satisfy the frozen prerequisites; "
            "E7-B was not run."
        ),
    }

with open(ROOT / "E7_final_verdict.json", "w") as f:
    json.dump(verdict, f, indent=2)

print(json.dumps(verdict, indent=2))

{
  "experiment": "E7_Waterbirds_real_image_bridge_v1",
  "go": false,
  "first_lambda_pcollapse_ge_0_5": null,
  "checks": {
    "E7A_eligible": false
  },
  "interpretation": "Waterbirds did not satisfy the frozen prerequisites; E7-B was not run."
}


## 11. Figures

In [14]:
if E7_ELIGIBLE:
    # Figure 1 — selected angle
    fig, ax = plt.subplots(figsize=(8, 5))
    for aug, g in summary.groupby("augmentation"):
        g = g.sort_values("lambda")
        ax.plot(
            g["lambda"],
            g["median_abs_theta_deg"],
            marker="o",
            label=aug,
        )
    ax.axhline(
        CFG["collapse_angle_deg"],
        linestyle="--",
        linewidth=1,
    )
    ax.set_xlabel("lambda")
    ax.set_ylabel("Median |theta| (degrees)")
    ax.set_title("E7 Waterbirds: semantic-to-nuisance trajectory")
    ax.legend()
    fig.tight_layout()
    fig.savefig(ROOT / "E7_angle_vs_lambda.png", dpi=220)
    plt.show()

    # Figure 2 — collapse probability
    fig, ax = plt.subplots(figsize=(8, 5))
    for aug, g in summary.groupby("augmentation"):
        g = g.sort_values("lambda")
        ax.plot(
            g["lambda"],
            g["p_nuisance_collapse"],
            marker="o",
            label=aug,
        )
    ax.axhline(0.5, linestyle="--", linewidth=1)
    ax.set_xlabel("lambda")
    ax.set_ylabel("P(|theta| >= 80 degrees)")
    ax.set_title("E7 Waterbirds: nuisance-collapse probability")
    ax.legend()
    fig.tight_layout()
    fig.savefig(ROOT / "E7_collapse_probability.png", dpi=220)
    plt.show()

    # Figure 3 — semantic/nuisance tradeoff under misalignment
    mis = summary[
        summary["augmentation"] == "misaligned"
    ].sort_values("lambda")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(
        mis["lambda"],
        mis["mean_test_semantic_acc"],
        marker="o",
        label="semantic accuracy",
    )
    ax.plot(
        mis["lambda"],
        mis["mean_test_nuisance_acc"],
        marker="o",
        label="background accuracy",
    )
    ax.set_xlabel("lambda")
    ax.set_ylabel("Mean held-out test accuracy")
    ax.set_title("E7 Waterbirds: semantic vs nuisance")
    ax.legend()
    fig.tight_layout()
    fig.savefig(ROOT / "E7_accuracy_tradeoff.png", dpi=220)
    plt.show()

## 12. Run manifest and export ZIP

In [15]:
manifest = {
    "experiment_id": CFG["experiment_id"],
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "metadata_path": str(META_PATH),
    "data_root": str(DATA_ROOT),
    "metadata_rows": int(len(metadata)),
    "config": CFG,
}

with open(ROOT / "run_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

zip_path = Path(
    "/kaggle/working/E7_Waterbirds_real_image_bridge_outputs.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    for p in ROOT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(ROOT))

print("ZIP:", zip_path)
print("Files:")
for p in sorted(ROOT.iterdir()):
    print(" -", p.name)

ZIP: /kaggle/working/E7_Waterbirds_real_image_bridge_outputs.zip
Files:
 - E7A_eligibility_by_seed.csv
 - E7A_eligibility_summary.json
 - E7_final_verdict.json
 - config.json
 - run_manifest.json


## 13. Display everything important directly in Kaggle

This final cell makes the experiment readable even if the ZIP is not downloaded immediately.

In [16]:
from IPython.display import display, Image as IPyImage

print("=" * 90)
print("E7 WATERBIRDS — FINAL OUTPUT REVIEW")
print("=" * 90)

print("\n[1] E7-A ELIGIBILITY BY SEED")
display(pd.read_csv(ROOT / "E7A_eligibility_by_seed.csv"))

print("\n[2] E7-A ELIGIBILITY SUMMARY")
with open(ROOT / "E7A_eligibility_summary.json") as f:
    print(json.dumps(json.load(f), indent=2))

if (ROOT / "E7B_summary.csv").exists():
    print("\n[3] E7-B AGGREGATE SUMMARY")
    display(pd.read_csv(ROOT / "E7B_summary.csv"))
else:
    print("\n[3] E7-B was not run.")

if (ROOT / "E7B_seed_level_results.csv").exists():
    print("\n[4] E7-B SEED-LEVEL RESULTS")
    df = pd.read_csv(ROOT / "E7B_seed_level_results.csv")
    print("Rows:", len(df))
    display(df)

print("\n[5] FINAL E7 VERDICT")
with open(ROOT / "E7_final_verdict.json") as f:
    print(json.dumps(json.load(f), indent=2))

print("\n[6] FIGURES")
for name in [
    "E7_angle_vs_lambda.png",
    "E7_collapse_probability.png",
    "E7_accuracy_tradeoff.png",
]:
    p = ROOT / name
    if p.exists():
        print("\n", name)
        display(IPyImage(filename=str(p)))

print("\n[7] SAVED FILES")
for p in sorted(ROOT.iterdir()):
    if p.is_file():
        print(f"{p.name:45s} {p.stat().st_size / 1024:10.2f} KB")

zip_path = Path(
    "/kaggle/working/E7_Waterbirds_real_image_bridge_outputs.zip"
)
print("\n[8] ZIP STATUS")
if zip_path.exists():
    print("ZIP created successfully.")
    print("Path:", zip_path)
    print(f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB")
else:
    print("ZIP not found.")

print("\n" + "=" * 90)
print("END OF E7 REVIEW")
print("=" * 90)

E7 WATERBIRDS — FINAL OUTPUT REVIEW

[1] E7-A ELIGIBILITY BY SEED


,seed,dev_n_per_group,test_n_per_group,M_sem,M_nui,M_nui_over_M_sem,semantic_endpoint_acc,semantic_endpoint_ari,nuisance_endpoint_acc,nuisance_endpoint_ari,...,nuisance_residual_fraction,semantic_stronger,semantic_acc_ok,semantic_ari_ok,nuisance_probe_ok,competitive_ratio_ok,baseline_theta_ok,misalignment_ok,aligned_control_ok,seed_eligible
0,11,189,642,0.773766,0.630435,0.814762,0.891534,0.612684,0.917989,0.698461,...,0.986606,True,True,True,True,True,True,False,False,False
1,22,189,642,0.772482,0.672474,0.870537,0.894180,0.621009,0.936508,0.761841,...,0.982150,True,True,True,True,True,True,True,False,False
2,33,189,642,0.769404,0.646886,0.840762,0.896825,0.629391,0.931217,0.743452,...,0.987390,True,True,True,True,True,True,False,False,False
3,44,189,642,0.777173,0.677205,0.871370,0.902116,0.646322,0.944444,0.789845,...,0.987424,True,True,True,True,True,True,True,False,False
4,55,189,642,0.772744,0.665646,0.861405,0.894180,0.621009,0.931217,0.743452,...,0.988355,True,True,True,True,True,True,True,False,False



[2] E7-A ELIGIBILITY SUMMARY
{
  "eligible": false,
  "seed_pass_fraction": 0.0,
  "required_fraction": 0.6,
  "n_pass": 0,
  "n_total": 5
}

[3] E7-B was not run.

[5] FINAL E7 VERDICT
{
  "experiment": "E7_Waterbirds_real_image_bridge_v1",
  "go": false,
  "first_lambda_pcollapse_ge_0_5": null,
  "checks": {
    "E7A_eligible": false
  },
  "interpretation": "Waterbirds did not satisfy the frozen prerequisites; E7-B was not run."
}

[6] FIGURES

[7] SAVED FILES
E7A_eligibility_by_seed.csv                         2.35 KB
E7A_eligibility_summary.json                        0.11 KB
E7_final_verdict.json                               0.25 KB
config.json                                         8.65 KB
run_manifest.json                                  10.84 KB

[8] ZIP STATUS
ZIP created successfully.
Path: /kaggle/working/E7_Waterbirds_real_image_bridge_outputs.zip
Size: 0.01 MB

END OF E7 REVIEW
